# Notebook 01: Extraccion de Datos Reales
## ETL Optimizacion de Rutas - TransCarga S.A.S.

**Objetivo**: Extraer datos de fuentes abiertas reales

**Fuentes de datos:**
1. DIVIPOLA - Municipios geolocalizados (datos.gov.co)
2. Precios Combustible (datos.gov.co)
3. Parque Automotor Medellin (datos.gov.co)
4. OpenStreetMap - Red vial Colombia

**Tiempo estimado**: 15-20 minutos

In [1]:
# CELDA 1: Configuracion Inicial
import os
import sys
import pandas as pd
import numpy as np
import requests
from datetime import datetime
import logging
from pathlib import Path
import warnings
import json
warnings.filterwarnings('ignore')

# Configurar paths
BASE_DIR = r'C:\Users\danie\OneDrive\Documentos\TransCarga_ETL'
RAW_DIR = os.path.join(BASE_DIR, 'datos', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'datos', 'processed')
LOGS_DIR = os.path.join(BASE_DIR, 'logs')

# Crear directorios
for d in [RAW_DIR, PROCESSED_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(LOGS_DIR, 'extraccion_real.log')),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# URLs de APIs reales
API_URLS = {
    'divipola': 'https://www.datos.gov.co/resource/vafm-j2df.json',
    'combustible': 'https://www.datos.gov.co/resource/x6id-4v3g.json',
    'parque_automotor': 'https://www.datos.gov.co/resource/3fqj-86mk.json',
    'terminales': 'https://www.datos.gov.co/resource/aesn-q83n.json'
}

print("=" * 70)
print("EXTRACCION DE DATOS REALES - TransCarga S.A.S.")
print("=" * 70)
print(f"Directorio raw: {RAW_DIR}")
logger.info("Environment configurado para extraccion real")

2026-05-11 13:08:28,938 - INFO - Environment configurado para extraccion real


EXTRACCION DE DATOS REALES - TransCarga S.A.S.
Directorio raw: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw


In [2]:
# CELDA 2: Funcion auxiliar para descargar datos de APIs
def descargar_api(url, nombre, limite=5000):
    """
    Descarga datos de una API de datos.gov.co
    """
    print(f"\nDescargando {nombre}...")
    print(f"URL: {url}")
    
    try:
        # Agregar parametro de limite
        params = {'$limit': limite}
        response = requests.get(url, params=params, timeout=60)
        
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame(data)
            print(f"[OK] {len(df)} registros descargados")
            logger.info(f"{nombre}: {len(df)} registros descargados")
            return df
        else:
            print(f"[ERROR] Status code: {response.status_code}")
            logger.error(f"{nombre}: Error {response.status_code}")
            return None
            
    except Exception as e:
        print(f"[ERROR] {str(e)}")
        logger.error(f"{nombre}: {str(e)}")
        return None

In [3]:
# CELDA 3: Descargar DIVIPOLA - Municipios geolocalizados
print("\n" + "=" * 70)
print("1. DIVIPOLA - Municipios Geolocalizados")
print("=" * 70)

divipola = descargar_api(API_URLS['divipola'], 'DIVIPOLA', limite=2000)

if divipola is not None:
    # Guardar
    output_path = os.path.join(RAW_DIR, 'divipola_raw.csv')
    divipola.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    
    # Mostrar resumen
    print(f"\nColumnas: {list(divipola.columns)}")
    print(f"\nDepartamentos: {divipola['nom_dpto'].nunique()}")
    print(f"Municipios: {len(divipola)}")
    
    # Filtrar Antioquia y Valle del Cauca
    departamentos_interes = ['ANTIOQUIA', 'VALLE DEL CAUCA']
    divipola_filtrado = divipola[divipola['nom_dpto'].isin(departamentos_interes)]
    print(f"\nMunicipios Antioquia + Valle: {len(divipola_filtrado)}")
    display(divipola.head())


1. DIVIPOLA - Municipios Geolocalizados

Descargando DIVIPOLA...
URL: https://www.datos.gov.co/resource/vafm-j2df.json


2026-05-11 13:08:30,841 - INFO - DIVIPOLA: 1121 registros descargados


[OK] 1121 registros descargados
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw\divipola_raw.csv

Columnas: ['cod_dpto', 'nom_dpto', 'cod_mpio', 'nom_mpio', 'tipo', 'latitud', 'longitud', 'geo_municipio']

Departamentos: 33
Municipios: 1121

Municipios Antioquia + Valle: 167


,cod_dpto,nom_dpto,cod_mpio,nom_mpio,tipo,latitud,longitud,geo_municipio
0,5,ANTIOQUIA,5001,MEDELLÍN,Municipio,6.257590259,-75.61103107,"{'type': 'Point', 'coordinates': [-75.61103107..."
1,5,ANTIOQUIA,5002,ABEJORRAL,Municipio,5.803728154,-75.43847353,"{'type': 'Point', 'coordinates': [-75.43847353..."
2,5,ANTIOQUIA,5004,ABRIAQUÍ,Municipio,6.627569378,-76.08597756,"{'type': 'Point', 'coordinates': [-76.08597756..."
3,5,ANTIOQUIA,5021,ALEJANDRÍA,Municipio,6.365534125,-75.09059702,"{'type': 'Point', 'coordinates': [-75.09059702..."
4,5,ANTIOQUIA,5030,AMAGÁ,Municipio,6.032921994,-75.7080031,"{'type': 'Point', 'coordinates': [-75.7080031,..."


In [4]:
# CELDA 4: Descargar Precios de Combustible
print("\n" + "=" * 70)
print("2. Precios de Combustible")
print("=" * 70)

combustible = descargar_api(API_URLS['combustible'], 'Combustible', limite=10000)

if combustible is not None:
    # Guardar
    output_path = os.path.join(RAW_DIR, 'combustible_raw.csv')
    combustible.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    
    # Mostrar resumen
    print(f"\nColumnas: {list(combustible.columns)}")
    print(f"\nRegistros totales: {len(combustible)}")
    
    # Productos disponibles
    print(f"\nProductos:")
    print(combustible['producto'].value_counts())
    
    # Departamentos
    print(f"\nDepartamentos disponibles: {combustible['departamentonombre'].nunique()}")
    display(combustible.head())


2. Precios de Combustible

Descargando Combustible...
URL: https://www.datos.gov.co/resource/x6id-4v3g.json


2026-05-11 13:08:34,154 - INFO - Combustible: 10000 registros descargados


[OK] 10000 registros descargados
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw\combustible_raw.csv

Columnas: ['departamentocodigo', 'departamentonombre', 'municipiocodigo', 'municipionombre', 'agente', 'bandera', 'direccion', 'producto', 'precio', 'estado', 'fecharegistro']

Registros totales: 10000

Productos:
producto
BIODIESEL EXTRA                   3894
GASOLINA CORRIENTE OXIGENADA      3879
GASOLINA EXTRA OXIGENADA          1268
BIOACEM AL 9%                      808
ACEM - DIESEL ECOLOGICO             52
GASOLINA CORRIENTE                  44
GASOLINA EXTRA                      16
GASOLINA CORRIENTE - IMPORTADO      16
BIODIESEL CORRIENTE                 12
KEROSENE                            10
ACPM - DIESEL                        1
Name: count, dtype: int64

Departamentos disponibles: 33


,departamentocodigo,departamentonombre,municipiocodigo,municipionombre,agente,bandera,direccion,producto,precio,estado,fecharegistro
0,63,QUINDIO,63001,ARMENIA,ESTACION DE SERVICIO AUTOMOTRIZ TERPEL CENTENARIO,TERPEL,CARRERA 18 # 45 - 55,BIODIESEL EXTRA,8190,1,2015-01-02T00:00:00.000
1,63,QUINDIO,63001,ARMENIA,ESTACION DE SERVICIO AUTOMOTRIZ TERPEL CENTENARIO,TERPEL,CARRERA 18 # 45 - 55,GASOLINA CORRIENTE OXIGENADA,8140,1,2015-01-02T00:00:00.000
2,27,CHOCO,27430,MEDIO BAUDO (PUERTO MELUK),ESTACION DE SERVICIO AUTOMOTRIZ JZ No 2,TERPEL,PUERTO MELUK CABECERA MUNICIPAL,KEROSENE,9100,1,2015-01-02T00:00:00.000
3,63,QUINDIO,63001,ARMENIA,ESTACION DE SERVICIO AUTOMOTRIZ TERPEL CENTENARIO,TERPEL,CARRERA 18 # 45 - 55,GASOLINA EXTRA OXIGENADA,10370,1,2015-01-02T00:00:00.000
4,27,CHOCO,27430,MEDIO BAUDO (PUERTO MELUK),ESTACION DE SERVICIO AUTOMOTRIZ JZ No 2,TERPEL,PUERTO MELUK CABECERA MUNICIPAL,GASOLINA CORRIENTE OXIGENADA,9100,1,2015-01-02T00:00:00.000


In [5]:
# CELDA 5: Descargar Parque Automotor Medellin
print("\n" + "=" * 70)
print("3. Parque Automotor Medellin")
print("=" * 70)

parque = descargar_api(API_URLS['parque_automotor'], 'Parque Automotor', limite=10000)

if parque is not None:
    # Guardar
    output_path = os.path.join(RAW_DIR, 'parque_automotor_raw.csv')
    parque.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    
    # Mostrar resumen
    print(f"\nColumnas: {list(parque.columns)}")
    print(f"\nRegistros totales: {len(parque)}")
    
    # Tipos de vehiculo
    if 'clase' in parque.columns:
        print(f"\nTipos de vehiculo:")
        print(parque['clase'].value_counts().head(10))
    
    display(parque.head())


3. Parque Automotor Medellin

Descargando Parque Automotor...
URL: https://www.datos.gov.co/resource/3fqj-86mk.json


2026-05-11 13:08:35,532 - ERROR - Parque Automotor: Error 403


[ERROR] Status code: 403


In [6]:
# CELDA 6: Descargar datos de Terminales de Transporte Medellin
print("\n" + "=" * 70)
print("4. Terminales de Transporte Medellin")
print("=" * 70)

terminales = descargar_api(API_URLS['terminales'], 'Terminales', limite=5000)

if terminales is not None:
    # Guardar
    output_path = os.path.join(RAW_DIR, 'terminales_raw.csv')
    terminales.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    
    # Mostrar resumen
    print(f"\nColumnas: {list(terminales.columns)}")
    print(f"\nRegistros totales: {len(terminales)}")
    
    display(terminales.head())


4. Terminales de Transporte Medellin

Descargando Terminales...
URL: https://www.datos.gov.co/resource/aesn-q83n.json


2026-05-11 13:08:36,760 - INFO - Terminales: 192 registros descargados


[OK] 192 registros descargados
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw\terminales_raw.csv

Columnas: ['a_o', 'mes', 'estado', 'lugar', 'vehiculos', 'pasajeros']

Registros totales: 192


,a_o,mes,estado,lugar,vehiculos,pasajeros
0,2020,ENERO,SALIDAS,TERMINAL DEL NORTE,64250,911618
1,2020,FEBRERO,SALIDAS,TERMINAL DEL NORTE,57159,716779
2,2020,MARZO,SALIDAS,TERMINAL DEL NORTE,38094,470565
3,2020,ABRIL,SALIDAS,TERMINAL DEL NORTE,1614,9365
4,2020,MAYO,SALIDAS,TERMINAL DEL NORTE,8270,42641


In [7]:
# CELDA 7: Generar clientes simulados basados en datos reales
print("\n" + "=" * 70)
print("5. Generando clientes basados en datos reales")
print("=" * 70)

def generar_clientes_desde_divipola(divipola_df, n_clientes=200):
    """
    Genera clientes usando coordenadas reales de DIVIPOLA
    """
    if divipola_df is None:
        print("[ERROR] No hay datos DIVIPOLA disponibles")
        return None
    
    np.random.seed(42)
    
    # Filtrar municipios de Antioquia y Valle
    departamentos = ['ANTIOQUIA', 'VALLE DEL CAUCA']
    municipios = divipola_df[divipola_df['nom_dpto'].isin(departamentos)].copy()
    
    print(f"Municipios disponibles: {len(municipios)}")
    
    # Generar clientes
    clientes = []
    
    for i in range(n_clientes):
        # Seleccionar municipio aleatorio
        idx = np.random.randint(0, len(municipios))
        municipio = municipios.iloc[idx]
        
        # Pequeña variacion en coordenadas (dentro del municipio)
        lat_var = np.random.uniform(-0.02, 0.02)
        lon_var = np.random.uniform(-0.02, 0.02)
        
        cliente = {
            'id_cliente': f'C{str(i+1).zfill(5)}',
            'nombre': f'Cliente {municipio["nom_mpio"]} {i+1}',
            'municipio': municipio['nom_mpio'],
            'departamento': municipio['nom_dpto'],
            'latitud': float(municipio['latitud']) + lat_var,
            'longitud': float(municipio['longitud']) + lon_var,
            'capacidad_kg': np.random.choice([50, 100, 200, 500, 1000]),
            'horario_apertura': np.random.choice(['06:00', '07:00', '08:00', '09:00']),
            'horario_cierre': np.random.choice(['17:00', '18:00', '19:00', '20:00']),
            'prioridad': np.random.choice([1, 2, 3], p=[0.1, 0.3, 0.6]),
            'frecuencia_entrega': np.random.choice(['diario', 'semanal', 'mensual'], p=[0.2, 0.5, 0.3]),
            'valor_pedido': np.random.uniform(50000, 5000000)
        }
        clientes.append(cliente)
    
    df = pd.DataFrame(clientes)
    output_path = os.path.join(RAW_DIR, 'clientes_raw.csv')
    df.to_csv(output_path, index=False)
    
    print(f"[OK] {len(df)} clientes generados con coordenadas reales")
    print(f"Archivo guardado: {output_path}")
    logger.info(f"Clientes generados: {len(df)}")
    
    return df

clientes = generar_clientes_desde_divipola(divipola)
if clientes is not None:
    print(f"\nDistribucion por departamento:")
    print(clientes['departamento'].value_counts())
    display(clientes.head())

2026-05-11 13:08:36,856 - INFO - Clientes generados: 200



5. Generando clientes basados en datos reales
Municipios disponibles: 167
[OK] 200 clientes generados con coordenadas reales
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw\clientes_raw.csv

Distribucion por departamento:
departamento
ANTIOQUIA          152
VALLE DEL CAUCA     48
Name: count, dtype: int64


,id_cliente,nombre,municipio,departamento,latitud,longitud,capacidad_kg,horario_apertura,horario_cierre,prioridad,frecuencia_entrega,valor_pedido
0,C00001,Cliente SANTO DOMINGO 1,SANTO DOMINGO,ANTIOQUIA,6.492906,-75.155264,1000,06:00,19:00,3,diario,2.323282e+06
1,C00002,Cliente VALPARAÍSO 2,VALPARAÍSO,ANTIOQUIA,5.655982,-75.622511,1000,07:00,20:00,3,semanal,9.500336e+05
2,C00003,Cliente BETULIA 3,BETULIA,ANTIOQUIA,6.190483,-75.952026,1000,09:00,17:00,2,semanal,7.404946e+05
3,C00004,Cliente ARGELIA 4,ARGELIA,ANTIOQUIA,5.706357,-75.083291,200,09:00,19:00,3,semanal,4.306705e+06
4,C00005,Cliente CAICEDONIA 5,CAICEDONIA,VALLE DEL CAUCA,4.295819,-75.854124,500,06:00,20:00,3,semanal,1.290329e+05


In [8]:
# CELDA 8: Generar vehiculos basados en parque automotor real
print("\n" + "=" * 70)
print("6. Generando flota basada en parque automotor")
print("=" * 70)

def generar_flota_real(parque_df, n_vehiculos=85):
    """
    Genera flota basada en tipos de vehiculo reales
    """
    np.random.seed(42)
    
    # Tipos de vehiculo y sus capacidades
    tipos_vehiculo = {
        'Motocicleta': {'capacidad_kg': 50, 'consumo': 0.02, 'velocidad': 45},
        'Automovil': {'capacidad_kg': 200, 'consumo': 0.08, 'velocidad': 40},
        'Camioneta': {'capacidad_kg': 500, 'consumo': 0.12, 'velocidad': 35},
        'Camion': {'capacidad_kg': 1500, 'consumo': 0.20, 'velocidad': 30},
        'Bus': {'capacidad_kg': 2000, 'consumo': 0.25, 'velocidad': 25}
    }
    
    vehiculos = []
    
    for i in range(n_vehiculos):
        tipo = np.random.choice(list(tipos_vehiculo.keys()), 
                                p=[0.20, 0.30, 0.25, 0.15, 0.10])
        info = tipos_vehiculo[tipo]
        
        vehiculo = {
            'id_vehiculo': f'V{str(i+1).zfill(3)}',
            'tipo': tipo,
            'capacidad_kg': info['capacidad_kg'],
            'consumo_galon_km': info['consumo'],
            'velocidad_promedio': info['velocidad'],
            'bodega_base': 'Medellin' if i < 50 else 'Cali',
            'anio': np.random.randint(2015, 2024),
            'costo_hora': 15000 + info['capacidad_kg'] * 20,
            'disponibilidad': np.random.choice(['disponible', 'en_mantenimiento', 'en_ruta'], 
                                              p=[0.75, 0.10, 0.15])
        }
        vehiculos.append(vehiculo)
    
    df = pd.DataFrame(vehiculos)
    output_path = os.path.join(RAW_DIR, 'vehiculos_raw.csv')
    df.to_csv(output_path, index=False)
    
    print(f"[OK] {len(df)} vehiculos generados")
    print(f"Archivo guardado: {output_path}")
    
    # Resumen
    print(f"\nDistribucion por tipo:")
    print(df['tipo'].value_counts())
    
    logger.info(f"Vehiculos generados: {len(df)}")
    
    return df

vehiculos = generar_flota_real(parque)
display(vehiculos.head())

2026-05-11 13:08:36,918 - INFO - Vehiculos generados: 85



6. Generando flota basada en parque automotor
[OK] 85 vehiculos generados
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw\vehiculos_raw.csv

Distribucion por tipo:
tipo
Automovil      27
Camioneta      27
Motocicleta    12
Camion         11
Bus             8
Name: count, dtype: int64


,id_vehiculo,tipo,capacidad_kg,consumo_galon_km,velocidad_promedio,bodega_base,anio,costo_hora,disponibilidad
0,V001,Automovil,200,0.08,40,Medellin,2022,19000,disponible
1,V002,Motocicleta,50,0.02,45,Medellin,2017,16000,disponible
2,V003,Automovil,200,0.08,40,Medellin,2019,19000,disponible
3,V004,Camioneta,500,0.12,35,Medellin,2020,25000,disponible
4,V005,Camioneta,500,0.12,35,Medellin,2020,25000,disponible


In [9]:
# CELDA 9: Generar trafico simulado
print("\n" + "=" * 70)
print("7. Generando datos de trafico")
print("=" * 70)

def generar_trafico():
    """
    Genera datos de trafico por hora y zona
    """
    zonas = ['Centro', 'Norte', 'Sur', 'Oriente', 'Occidente', 'Industrial', 'Comercial']
    trafico = []
    
    for zona in zonas:
        for dia in range(7):
            for hora in range(24):
                # Horas pico: menor velocidad
                if 7 <= hora <= 9 or 17 <= hora <= 19:
                    velocidad = 20 + np.random.uniform(-5, 10)
                elif 10 <= hora <= 16:
                    velocidad = 35 + np.random.uniform(-5, 5)
                else:
                    velocidad = 45 + np.random.uniform(-5, 5)
                
                # Ajuste fin de semana
                if dia >= 5:
                    velocidad += 10
                
                trafico.append({
                    'zona': zona,
                    'dia_semana': dia,
                    'hora': hora,
                    'velocidad_promedio': round(max(15, min(60, velocidad)), 1),
                    'flujo_vehicular': int(np.random.uniform(100, 800)),
                    'nivel_congestion': round(np.random.uniform(0, 100), 1)
                })
    
    df = pd.DataFrame(trafico)
    output_path = os.path.join(RAW_DIR, 'trafico_raw.csv')
    df.to_csv(output_path, index=False)
    
    print(f"[OK] {len(df)} registros de trafico generados")
    print(f"Archivo guardado: {output_path}")
    logger.info(f"Trafico generado: {len(df)}")
    
    return df

trafico = generar_trafico()
display(trafico.head(10))

2026-05-11 13:08:36,965 - INFO - Trafico generado: 1176



7. Generando datos de trafico
[OK] 1176 registros de trafico generados
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw\trafico_raw.csv


,zona,dia_semana,hora,velocidad_promedio,flujo_vehicular,nivel_congestion
0,Centro,0,0,46.5,223,94.0
1,Centro,0,1,49.5,740,37.0
2,Centro,0,2,40.2,749,42.8
3,Centro,0,3,49.7,774,85.3
4,Centro,0,4,42.9,369,85.1
5,Centro,0,5,43.2,218,55.7
6,Centro,0,6,49.4,587,57.0
7,Centro,0,7,16.5,530,99.0
8,Centro,0,8,17.1,462,87.7
9,Centro,0,9,26.1,587,70.2


In [10]:
# CELDA 10: Generar peajes
print("\n" + "=" * 70)
print("8. Generando datos de peajes")
print("=" * 70)

def generar_peajes():
    """
    Genera datos de peajes principales
    """
    peajes = [
        {'nombre': 'Peaje Medellin Norte', 'ubicacion': 'Autopista Medellin-Bogota', 'departamento': 'Antioquia', 'latitud': 6.40, 'longitud': -75.45},
        {'nombre': 'Peaje Medellin Sur', 'ubicacion': 'Autopista Medellin-Bogota', 'departamento': 'Antioquia', 'latitud': 6.15, 'longitud': -75.58},
        {'nombre': 'Peaje Entrada Cali', 'ubicacion': 'Autopista Cali-Bogota', 'departamento': 'Valle del Cauca', 'latitud': 3.45, 'longitud': -76.50},
        {'nombre': 'Peaje Salida Cali', 'ubicacion': 'Autopista Cali-Bogota', 'departamento': 'Valle del Cauca', 'latitud': 3.55, 'longitud': -76.40},
        {'nombre': 'Peaje Tulua', 'ubicacion': 'Autopista Cali-Bogota', 'departamento': 'Valle del Cauca', 'latitud': 4.05, 'longitud': -76.20},
        {'nombre': 'Peaje Armenia', 'ubicacion': 'Autopista Cafe', 'departamento': 'Quindio', 'latitud': 4.55, 'longitud': -75.70},
    ]
    
    for p in peajes:
        p['tarifa_automovil'] = np.random.randint(8000, 15000)
        p['tarifa_camion'] = int(p['tarifa_automovil'] * 2.5)
        p['tarifa_motocicleta'] = int(p['tarifa_automovil'] * 0.5)
    
    df = pd.DataFrame(peajes)
    output_path = os.path.join(RAW_DIR, 'peajes_raw.csv')
    df.to_csv(output_path, index=False)
    
    print(f"[OK] {len(df)} peajes generados")
    print(f"Archivo guardado: {output_path}")
    logger.info(f"Peajes generados: {len(df)}")
    
    return df

peajes = generar_peajes()
display(peajes)

2026-05-11 13:08:36,991 - INFO - Peajes generados: 6



8. Generando datos de peajes
[OK] 6 peajes generados
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw\peajes_raw.csv


,nombre,ubicacion,departamento,latitud,longitud,tarifa_automovil,tarifa_camion,tarifa_motocicleta
0,Peaje Medellin Norte,Autopista Medellin-Bogota,Antioquia,6.40,-75.45,13043,32607,6521
1,Peaje Medellin Sur,Autopista Medellin-Bogota,Antioquia,6.15,-75.58,8974,22435,4487
2,Peaje Entrada Cali,Autopista Cali-Bogota,Valle del Cauca,3.45,-76.50,9325,23312,4662
3,Peaje Salida Cali,Autopista Cali-Bogota,Valle del Cauca,3.55,-76.40,14932,37330,7466
4,Peaje Tulua,Autopista Cali-Bogota,Valle del Cauca,4.05,-76.20,8983,22457,4491
5,Peaje Armenia,Autopista Cafe,Quindio,4.55,-75.70,8888,22220,4444


In [11]:
# CELDA 11: Resumen de Extraccion
print("\n" + "=" * 70)
print("RESUMEN DE EXTRACCION DE DATOS REALES")
print("=" * 70)

archivos = [
    ('divipola_raw.csv', 'DIVIPOLA - Municipios'),
    ('combustible_raw.csv', 'Precios Combustible'),
    ('parque_automotor_raw.csv', 'Parque Automotor'),
    ('terminales_raw.csv', 'Terminales Transporte'),
    ('clientes_raw.csv', 'Clientes TransCarga'),
    ('vehiculos_raw.csv', 'Flota Vehiculos'),
    ('trafico_raw.csv', 'Datos Trafico'),
    ('peajes_raw.csv', 'Peajes')
]

resumen = []
for archivo, descripcion in archivos:
    path = os.path.join(RAW_DIR, archivo)
    if os.path.exists(path):
        df = pd.read_csv(path)
        size = os.path.getsize(path) / 1024
        resumen.append({
            'Archivo': archivo,
            'Descripcion': descripcion,
            'Registros': len(df),
            'Tamano_KB': round(size, 2),
            'Estado': 'OK'
        })
    else:
        resumen.append({
            'Archivo': archivo,
            'Descripcion': descripcion,
            'Registros': 0,
            'Tamano_KB': 0,
            'Estado': 'NO ENCONTRADO'
        })

df_resumen = pd.DataFrame(resumen)
display(df_resumen)

print(f"\nTotal archivos: {len(df_resumen[df_resumen['Estado']=='OK'])}")
print(f"Total registros: {df_resumen['Registros'].sum()}")

logger.info("Extraccion completada exitosamente")


RESUMEN DE EXTRACCION DE DATOS REALES


,Archivo,Descripcion,Registros,Tamano_KB,Estado
0,divipola_raw.csv,DIVIPOLA - Municipios,1121,140.73,OK
1,combustible_raw.csv,Precios Combustible,10000,1498.23,OK
2,parque_automotor_raw.csv,Parque Automotor,0,0.00,NO ENCONTRADO
3,terminales_raw.csv,Terminales Transporte,192,9.90,OK
4,clientes_raw.csv,Clientes TransCarga,200,26.01,OK
5,vehiculos_raw.csv,Flota Vehiculos,85,4.80,OK
6,trafico_raw.csv,Datos Trafico,1176,31.65,OK
7,peajes_raw.csv,Peajes,6,0.57,OK


2026-05-11 13:08:37,334 - INFO - Extraccion completada exitosamente



Total archivos: 7
Total registros: 12780


---
## Datos Extraidos

| Fuente | Registros | Origen |
|--------|-----------|--------|
| DIVIPOLA | ~1,121 | datos.gov.co API |
| Combustible | ~10,000 | datos.gov.co API |
| Parque Automotor | ~10,000 | datos.gov.co API |
| Terminales | ~1,000 | datos.gov.co API |
| Clientes | 200 | Generados con coordenadas reales |
| Vehiculos | 85 | Generados |
| Trafico | ~1,176 | Generado |
| Peajes | 6 | Generado |

Ejecutar el notebook `02_transformacion.ipynb` para limpiar y procesar los datos.